### Smart Outcome Predictor: Ensemble Machine Learning System
**Red & White Skill Education — Project PR. 5**

**Tasks:**
- **Classification Task:** Predict whether a student completes an online course (`completion_status`: 0 or 1)
- **Regression Task:** Predict student final performance score (`final_score`: 0 to 100)
- **Techniques Explored:** Single Base Models, Bagging, AdaBoost, XGBoost, Hard/Soft Voting, Stacking
- **Artifacts Exported:** Model files (`.pkl`) and metadata/metrics (`.json`) inside the `model/` folder.


## Part A: Conceptual Foundation (Theory)

### 1. What are Ensemble Learning Methods and why are they effective?
Ensemble learning methods combine predictions from multiple individual models (weak or strong learners) to produce a single aggregated prediction. They are effective because aggregation reduces individual model errors (variance, bias, or noise), leading to superior generalization, higher stability, and lower susceptibility to overfitting compared to any single constituent model.

### 2. Difference between Bagging, Boosting, and Voting
- **Bagging (Bootstrap Aggregating):** Trains multiple base models in parallel on random bootstrap subsets of the training data with replacement and averages (or majority votes) their predictions to reduce variance.
- **Boosting:** Trains base models sequentially where each successive learner focuses on the mistakes (residuals or weighted errors) made by prior models, primarily reducing bias.
- **Voting:** Combines diverse, independently trained models (often of completely different algorithms) using majority class voting (hard voting) or predicted probability averaging (soft voting).

### 3. Bias-Variance Trade-off & How Ensembles Address It
The bias-variance trade-off describes the tension between model underfitting (high bias, overly simplistic) and overfitting (high variance, overly sensitive to training noise). Ensemble methods mitigate this trade-off effectively:
- **Bagging & Random Forests** aggregate diverse high-variance trees to dramatically reduce variance without increasing bias.
- **Boosting** starts with weak, high-bias learners and iteratively minimizes bias while keeping model complexity controlled through learning rates and regularization.

### 4. Voting Classifier vs. Stacking Ensemble
- **Voting Classifier:** Combines predictions of multiple models using predetermined statistical heuristics (mode for hard voting, mean/weighted probability for soft voting) without training an additional learning layer.
- **Stacking Ensemble:** Employs a meta-model (level-1 learner) trained directly on the cross-validated out-of-fold predictions produced by the base models (level-0 learners), allowing the system to learn which base models perform best in specific feature regions.

### 5. AdaBoost vs. XGBoost (Conceptual Comparison)
- **AdaBoost (Adaptive Boosting):** Increases the sample weights of misclassified instances at each step, forcing new decision stumps (shallow trees) to prioritize difficult cases, aggregating predictions via weighted voting.
- **XGBoost (Extreme Gradient Boosting):** Fits consecutive trees to the negative gradient (residuals) of a differentiable loss function using second-order Taylor expansion (gradients & Hessians), incorporating built-in L1/L2 tree regularization, sparsity awareness, and parallel tree-building heuristics.


In [1]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    BaggingClassifier, BaggingRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    VotingClassifier, StackingClassifier,
    StackingRegressor
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, root_mean_squared_error, r2_score, classification_report
)

# XGBoost
from xgboost import XGBClassifier, XGBRegressor

# Plot settings
sns.set_theme(style="whitegrid")
np.random.seed(42)


## Part B: Dataset Understanding & Preparation
We load the base EdTech dataset `Smart_Outcome_Predictor_Dataset_5200.xlsx` (with CSV support) containing student demographics, platform engagement metrics, and dual targets:
1. `completion_status` (Classification: 0 or 1)
2. `final_score` (Regression: 0 to 100)


In [2]:
# Load the provided dataset from either the project root or notebooks/ working directory
DATASET_NAME = "Smart_Outcome_Predictor_Dataset_5200"
data_candidates = [
    os.path.join(os.getcwd(), "data", f"{DATASET_NAME}.xlsx"),
    os.path.join(os.getcwd(), "..", "data", f"{DATASET_NAME}.xlsx"),
    os.path.join(os.getcwd(), "data", f"{DATASET_NAME}.csv"),
    os.path.join(os.getcwd(), "..", "data", f"{DATASET_NAME}.csv"),
]
data_path = next((path for path in data_candidates if os.path.isfile(path)), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Expected Smart_Outcome_Predictor_Dataset_5200.xlsx "
        "or .csv in the project's data/ directory."
    )

if data_path.lower().endswith(".xlsx"):
    df = pd.read_excel(data_path)
else:
    df = pd.read_csv(data_path)

print("Dataset path:", os.path.abspath(data_path))

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# Handle missing values in numerical columns
for col in ['time_spent_hours', 'avg_quiz_score', 'attendance_rate']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Drop identifier and date columns not useful directly for model training
drop_cols = [c for c in ['student_id', 'course_start_date'] if c in df.columns]
df_cleaned = df.drop(columns=drop_cols)

# Encode categorical variables using pd.get_dummies
cat_cols = ['country_region', 'device_type', 'education_background', 'course_level', 'course_category']
cat_cols = [c for c in cat_cols if c in df_cleaned.columns]
df_encoded = pd.get_dummies(df_cleaned, columns=cat_cols, drop_first=True)

print("Processed dataset shape after encoding:", df_encoded.shape)
df_encoded.head()


Dataset path: d:\Smart_Outcome_ Predictor\data\Smart_Outcome_Predictor_Dataset_5200.xlsx
Dataset shape: (5200, 19)
Columns: ['student_id', 'age', 'country_region', 'device_type', 'education_background', 'course_level', 'course_category', 'course_start_date', 'week_of_year', 'sessions', 'time_spent_hours', 'videos_watched', 'quiz_attempts', 'assignments_submitted', 'forum_posts', 'avg_quiz_score', 'attendance_rate', 'completion_status', 'final_score']
Processed dataset shape after encoding: (5200, 27)


,age,week_of_year,sessions,time_spent_hours,videos_watched,quiz_attempts,assignments_submitted,forum_posts,avg_quiz_score,attendance_rate,...,device_type_Tablet,education_background_HighSchool,education_background_Undergrad,education_background_WorkingPro,course_level_Beginner,course_level_Intermediate,course_category_Data,course_category_Design,course_category_Marketing,course_category_Programming
0,32,12,1,7.6,1,6,1,1,53.3,0.655,...,False,False,True,False,False,True,False,False,False,False
1,17,34,16,27.2,6,4,7,1,51.5,1.000,...,False,False,True,False,False,True,False,False,False,True
2,25,39,6,7.1,16,2,2,0,62.2,0.810,...,False,False,False,False,False,False,False,False,False,True
3,26,10,34,22.1,57,9,6,0,59.3,0.875,...,False,False,True,False,True,False,False,True,False,False
4,26,12,22,32.3,41,9,2,0,65.1,0.814,...,True,False,False,True,False,False,False,False,False,False


In [3]:
# Feature and Target Split
feature_cols = [c for c in df_encoded.columns if c not in ['completion_status', 'final_score']]

X = df_encoded[feature_cols]
y_clf = df_encoded['completion_status']
y_reg = df_encoded['final_score']

# Train-Test Split (Classification & Regression)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Standard Scaling
scaler = StandardScaler()
X_train_clf_scaled = scaler.fit_transform(X_train_clf)
X_test_clf_scaled = scaler.transform(X_test_clf)

X_train_reg_scaled = scaler.fit_transform(X_train_reg)
X_test_reg_scaled = scaler.transform(X_test_reg)

print("Preprocessing complete!")
print("Train shapes:", X_train_clf.shape, X_train_reg.shape)
print("Test shapes:", X_test_clf.shape, X_test_reg.shape)


Preprocessing complete!
Train shapes: (4160, 25) (4160, 25)
Test shapes: (1040, 25) (1040, 25)


## Part C: Bagging (Bootstrap Aggregating)
Bagging creates multiple bootstrap replicas of the training data and trains an independent estimator on each replica.
Below, we evaluate single base decision trees vs. Bagging ensembles for both classification and regression.


In [4]:
# 1. Base Decision Tree vs Bagging Classifier
base_dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
base_dt_clf.fit(X_train_clf, y_train_clf)
dt_pred_clf = base_dt_clf.predict(X_test_clf)
dt_prob_clf = base_dt_clf.predict_proba(X_test_clf)[:, 1]

bag_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
bag_clf.fit(X_train_clf, y_train_clf)
bag_pred_clf = bag_clf.predict(X_test_clf)
bag_prob_clf = bag_clf.predict_proba(X_test_clf)[:, 1]

print("--- CLASSIFICATION: Base Tree vs Bagging ---")
print(f"Base DT Accuracy : {accuracy_score(y_test_clf, dt_pred_clf):.4f} | ROC-AUC: {roc_auc_score(y_test_clf, dt_prob_clf):.4f}")
print(f"Bagging Accuracy : {accuracy_score(y_test_clf, bag_pred_clf):.4f} | ROC-AUC: {roc_auc_score(y_test_clf, bag_prob_clf):.4f}")

# 2. Base Decision Tree vs Bagging Regressor
base_dt_reg = DecisionTreeRegressor(max_depth=6, random_state=42)
base_dt_reg.fit(X_train_reg, y_train_reg)
dt_pred_reg = base_dt_reg.predict(X_test_reg)

bag_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=6, random_state=42),
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)
bag_reg.fit(X_train_reg, y_train_reg)
bag_pred_reg = bag_reg.predict(X_test_reg)

print("
--- REGRESSION: Base Tree vs Bagging ---")
print(f"Base DT RMSE : {root_mean_squared_error(y_test_reg, dt_pred_reg):.4f} | R2: {r2_score(y_test_reg, dt_pred_reg):.4f}")
print(f"Bagging RMSE : {root_mean_squared_error(y_test_reg, bag_pred_reg):.4f} | R2: {r2_score(y_test_reg, bag_pred_reg):.4f}")


SyntaxError: unterminated string literal (detected at line 35) (1150019980.py, line 35)

## Part D: Boosting Algorithms (AdaBoost & XGBoost)
Boosting trains trees sequentially. AdaBoost adjusts sample weights based on errors; XGBoost fits successive trees to the pseudo-residuals using gradient boosting principles.


In [ ]:
# AdaBoost Models
ada_clf = AdaBoostClassifier(n_estimators=50, learning_rate=0.8, random_state=42)
ada_clf.fit(X_train_clf, y_train_clf)
ada_pred_clf = ada_clf.predict(X_test_clf)
ada_prob_clf = ada_clf.predict_proba(X_test_clf)[:, 1]

ada_reg = AdaBoostRegressor(n_estimators=50, learning_rate=0.8, random_state=42)
ada_reg.fit(X_train_reg, y_train_reg)
ada_pred_reg = ada_reg.predict(X_test_reg)

# XGBoost Models
xgb_clf = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, subsample=0.8, random_state=42, eval_metric='logloss')
xgb_clf.fit(X_train_clf, y_train_clf)
xgb_pred_clf = xgb_clf.predict(X_test_clf)
xgb_prob_clf = xgb_clf.predict_proba(X_test_clf)[:, 1]

xgb_reg = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.08, subsample=0.8, random_state=42)
xgb_reg.fit(X_train_reg, y_train_reg)
xgb_pred_reg = xgb_reg.predict(X_test_reg)

print("Boosting Models Trained Successfully!")


## Part E: Voting & Stacking Ensembles
- **Voting Classifier:** We evaluate both **Hard Voting** (majority label) and **Soft Voting** (argmax of average predicted probabilities).
- **Stacking:** Base models pass out-of-fold predictions to a meta-model (Logistic Regression for classification, Ridge for regression).


In [ ]:
# Voting Classifier: Hard vs Soft
estimators_clf = [
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=50, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.1, random_state=42, eval_metric='logloss'))
]

hard_voting_clf = VotingClassifier(estimators=estimators_clf, voting='hard')
hard_voting_clf.fit(X_train_clf, y_train_clf)
hard_pred_clf = hard_voting_clf.predict(X_test_clf)

soft_voting_clf = VotingClassifier(estimators=estimators_clf, voting='soft')
soft_voting_clf.fit(X_train_clf, y_train_clf)
soft_pred_clf = soft_voting_clf.predict(X_test_clf)
soft_prob_clf = soft_voting_clf.predict_proba(X_test_clf)[:, 1]

print(f"Hard Voting Accuracy: {accuracy_score(y_test_clf, hard_pred_clf):.4f}")
print(f"Soft Voting Accuracy: {accuracy_score(y_test_clf, soft_pred_clf):.4f} | ROC-AUC: {roc_auc_score(y_test_clf, soft_prob_clf):.4f}")

# Stacking Classifier
stack_clf = StackingClassifier(
    estimators=estimators_clf,
    final_estimator=LogisticRegression(random_state=42),
    cv=5
)
stack_clf.fit(X_train_clf, y_train_clf)
stack_pred_clf = stack_clf.predict(X_test_clf)
stack_prob_clf = stack_clf.predict_proba(X_test_clf)[:, 1]

# Stacking Regressor
estimators_reg = [
    ('dt', DecisionTreeRegressor(max_depth=5, random_state=42)),
    ('ada', AdaBoostRegressor(n_estimators=50, random_state=42)),
    ('xgb', XGBRegressor(n_estimators=80, max_depth=3, learning_rate=0.1, random_state=42))
]

stack_reg = StackingRegressor(
    estimators=estimators_reg,
    final_estimator=Ridge(),
    cv=5
)
stack_reg.fit(X_train_reg, y_train_reg)
stack_pred_reg = stack_reg.predict(X_test_reg)

print("Stacking Classifier & Regressor Trained!")


## Part F: Model Evaluation & Comparison
Comprehensive benchmarking of all single and ensemble models for both classification and regression.


In [ ]:
# Evaluate Classification Models
clf_results = {
    'Single Decision Tree': {
        'Accuracy': accuracy_score(y_test_clf, dt_pred_clf),
        'Precision': precision_score(y_test_clf, dt_pred_clf),
        'Recall': recall_score(y_test_clf, dt_pred_clf),
        'F1-Score': f1_score(y_test_clf, dt_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, dt_prob_clf)
    },
    'Bagging Classifier': {
        'Accuracy': accuracy_score(y_test_clf, bag_pred_clf),
        'Precision': precision_score(y_test_clf, bag_pred_clf),
        'Recall': recall_score(y_test_clf, bag_pred_clf),
        'F1-Score': f1_score(y_test_clf, bag_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, bag_prob_clf)
    },
    'AdaBoost Classifier': {
        'Accuracy': accuracy_score(y_test_clf, ada_pred_clf),
        'Precision': precision_score(y_test_clf, ada_pred_clf),
        'Recall': recall_score(y_test_clf, ada_pred_clf),
        'F1-Score': f1_score(y_test_clf, ada_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, ada_prob_clf)
    },
    'XGBoost Classifier': {
        'Accuracy': accuracy_score(y_test_clf, xgb_pred_clf),
        'Precision': precision_score(y_test_clf, xgb_pred_clf),
        'Recall': recall_score(y_test_clf, xgb_pred_clf),
        'F1-Score': f1_score(y_test_clf, xgb_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, xgb_prob_clf)
    },
    'Hard Voting': {
        'Accuracy': accuracy_score(y_test_clf, hard_pred_clf),
        'Precision': precision_score(y_test_clf, hard_pred_clf),
        'Recall': recall_score(y_test_clf, hard_pred_clf),
        'F1-Score': f1_score(y_test_clf, hard_pred_clf),
        'ROC-AUC': np.nan
    },
    'Soft Voting': {
        'Accuracy': accuracy_score(y_test_clf, soft_pred_clf),
        'Precision': precision_score(y_test_clf, soft_pred_clf),
        'Recall': recall_score(y_test_clf, soft_pred_clf),
        'F1-Score': f1_score(y_test_clf, soft_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, soft_prob_clf)
    },
    'Stacking Classifier': {
        'Accuracy': accuracy_score(y_test_clf, stack_pred_clf),
        'Precision': precision_score(y_test_clf, stack_pred_clf),
        'Recall': recall_score(y_test_clf, stack_pred_clf),
        'F1-Score': f1_score(y_test_clf, stack_pred_clf),
        'ROC-AUC': roc_auc_score(y_test_clf, stack_prob_clf)
    }
}

df_clf_results = pd.DataFrame(clf_results).T.round(4)
print("=== CLASSIFICATION BENCHMARK ===")
display(df_clf_results)


In [ ]:
# Evaluate Regression Models
reg_results = {
    'Single Decision Tree': {
        'MAE': mean_absolute_error(y_test_reg, dt_pred_reg),
        'RMSE': root_mean_squared_error(y_test_reg, dt_pred_reg),
        'R2 Score': r2_score(y_test_reg, dt_pred_reg)
    },
    'Bagging Regressor': {
        'MAE': mean_absolute_error(y_test_reg, bag_pred_reg),
        'RMSE': root_mean_squared_error(y_test_reg, bag_pred_reg),
        'R2 Score': r2_score(y_test_reg, bag_pred_reg)
    },
    'AdaBoost Regressor': {
        'MAE': mean_absolute_error(y_test_reg, ada_pred_reg),
        'RMSE': root_mean_squared_error(y_test_reg, ada_pred_reg),
        'R2 Score': r2_score(y_test_reg, ada_pred_reg)
    },
    'XGBoost Regressor': {
        'MAE': mean_absolute_error(y_test_reg, xgb_pred_reg),
        'RMSE': root_mean_squared_error(y_test_reg, xgb_pred_reg),
        'R2 Score': r2_score(y_test_reg, xgb_pred_reg)
    },
    'Stacking Regressor': {
        'MAE': mean_absolute_error(y_test_reg, stack_pred_reg),
        'RMSE': root_mean_squared_error(y_test_reg, stack_pred_reg),
        'R2 Score': r2_score(y_test_reg, stack_pred_reg)
    }
}

df_reg_results = pd.DataFrame(reg_results).T.round(4)
print("=== REGRESSION BENCHMARK ===")
display(df_reg_results)


In [ ]:
# Comparative Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Classification plot (F1-Score & Accuracy)
df_clf_plot = df_clf_results[['Accuracy', 'F1-Score']]
df_clf_plot.plot(kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('Classification Model Performance Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0.7, 1.0)
axes[0].tick_params(axis='x', rotation=45)

# Regression plot (R2 Score)
df_reg_results['R2 Score'].plot(kind='bar', ax=axes[1], color='#2b5c8f')
axes[1].set_title('Regression Model Comparison (R2 Score)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('R2 Score')
axes[1].set_ylim(0.7, 1.0)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## Part G: Artifact Serialization & Export to `model/` Folder
In this step, we:
1. Ensure the `model/` folder exists.
2. Save the best trained classification and regression models as `.pkl` files.
3. Save feature column lists, normalization scaler, and benchmark summaries as `.json` metadata.


In [ ]:
# Define relative path to the 'model' directory from within 'notebooks/'
# If running inside notebooks/, the parent directory contains 'model/'
# We use os.path.abspath to ensure it correctly resolves whether run from 'notebooks' or project root
notebook_dir = os.getcwd()
if os.path.basename(notebook_dir) == "notebooks":
    model_dir = os.path.join("..", "model")
else:
    model_dir = "model"

os.makedirs(model_dir, exist_ok=True)

# 1. Save EXACTLY ONE .pkl file containing the trained models and preprocessor
model_artifacts = {
    "classifier": stack_clf,
    "regressor": xgb_reg,
    "scaler": scaler,
    "feature_names": feature_cols
}

pkl_filename = "smart_outcome_predictor.pkl"
pkl_path = os.path.join(model_dir, pkl_filename)

with open(pkl_path, "wb") as f:
    pickle.dump(model_artifacts, f)

print(f"Successfully saved .pkl file: {os.path.abspath(pkl_path)}")

# 2. Save EXACTLY ONE .json file containing metadata, parameters, and evaluation metrics
metadata = {
    "project_name": "Smart Outcome Predictor",
    "version": "1.0.0",
    "feature_columns": feature_cols,
    "target_variables": {
        "classification": "completion_status",
        "regression": "final_score"
    },
    "best_models": {
        "classification": "StackingClassifier (DecisionTree + AdaBoost + XGBoost -> LogisticRegression)",
        "regression": "XGBRegressor"
    },
    "classification_metrics": df_clf_results.to_dict(orient="index"),
    "regression_metrics": df_reg_results.to_dict(orient="index"),
    "sample_test_input": X.iloc[0].to_dict()
}

json_filename = "model_metadata.json"
json_path = os.path.join(model_dir, json_filename)

with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)

print(f"Successfully saved .json file: {os.path.abspath(json_path)}")

# Verification: check the files created inside the model folder
files_in_model = os.listdir(model_dir)
print(f"\nTotal files in '{model_dir}' folder: {len(files_in_model)}")
for file in files_in_model:
    print(f" - {file}")
